# 🚀 CrossTL CUDA Translation Demo

This notebook shows a small end-to-end CrossTL workflow:

**CUDA-style compute code → CrossGL IR → target backend code**

We’ll use CrossTL to take an example CUDA kernel, convert it into CrossGL’s intermediate representation, and then generate code for other backends such as Metal, HLSL, or GLSL.

The goal is to demonstrate the basic idea behind CrossTL:

> ✨ write once, port where you need.

This is a simple demo notebook, so the example is intentionally small. Feel free to modify the kernel, try different targets, break things, and see how the translator behaves 🧪


## Install CrossTL

Use the PyPI package for the clean public path. If you want the absolute latest repo version, switch `INSTALL_FROM_GITHUB` to `True`.

In [1]:
INSTALL_FROM_GITHUB = False

if INSTALL_FROM_GITHUB:
    !pip -q install -U git+https://github.com/CrossGL/crosstl.git
else:
    !pip -q install -U crosstl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 10.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 1. Write a tiny CUDA kernel

For the demo, we will use a simple `1D stencil / heat diffusion step` kernel.

It is small, recognizable, and easy to explain in a notebook.

In [7]:
from pathlib import Path

cuda_source = """
extern "C" __global__
void heat_step_1d(
    const float* input,
    float* output,
    int n,
    float alpha
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i >= n) {
        return;
    }

    // Keep boundary values unchanged
    if (i == 0 || i == n - 1) {
        output[i] = input[i];
        return;
    }

    // Simple 3-point stencil:
    // output[i] depends on the left neighbor, current value, and right neighbor
    output[i] =
        alpha * input[i - 1] +
        (1.0f - 2.0f * alpha) * input[i] +
        alpha * input[i + 1];
}
""".strip()

Path("steppy.cu").write_text(cuda_source)
print(cuda_source)

extern "C" __global__
void heat_step_1d(
    const float* input,
    float* output,
    int n,
    float alpha
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i >= n) {
        return;
    }

    // Keep boundary values unchanged
    if (i == 0 || i == n - 1) {
        output[i] = input[i];
        return;
    }

    // Simple 3-point stencil:
    // output[i] depends on the left neighbor, current value, and right neighbor
    output[i] =
        alpha * input[i - 1] +
        (1.0f - 2.0f * alpha) * input[i] +
        alpha * input[i + 1];
}


## 2. Import CUDA into CrossGL IR

CrossTL uses **CrossGL** as the shared intermediate representation.

So instead of building a separate translator for every possible pair, the flow becomes:

**CUDA → CrossGL IR → target backend**

In [9]:
import crosstl
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)

try:
    cgl_code = crosstl.translate("steppy.cu", backend="cgl", save_shader="outputs/steppy.cgl")
    print("✅ CUDA imported into CrossGL IR")
    print("\n--- CrossGL IR Preview ---\n")
    print(Path("outputs/steppy.cgl").read_text()[:2000])
except Exception as e:
    print("⚠️ Translation failed in this environment/package version.")
    print("This usually means the public package parser does not yet support this exact CUDA syntax.")
    print("Try switching INSTALL_FROM_GITHUB=True above, or simplify the input kernel.")
    raise e

✅ CUDA imported into CrossGL IR

--- CrossGL IR Preview ---

// CUDA to CrossGL conversion
// Kernel: heat_step_1d
@compute
@workgroup_size(1, 1, 1)  // Default workgroup size
fn heat_step_1d(
    @group(0) @binding(0) var<storage, read_write> input: array<f32>,
    @group(0) @binding(1) var<storage, read_write> output: array<f32>,
    i32 n,
    f32 alpha
) {
    let thread_id = gl_GlobalInvocationID;
    let block_id = gl_WorkGroupID;
    let thread_local_id = gl_LocalInvocationID;
    let block_dim = gl_WorkGroupSize;

    var i: i32 = ((gl_WorkGroupID.x * gl_WorkGroupSize.x) + gl_LocalInvocationID.x);
    if ((i >= n)) {
        return;
    }
    if (((i == 0) || (i == (n - 1)))) {
        output[i] = input[i];
        return;
    }
    output[i] = (((alpha * input[(i - 1)]) + ((1.0f - (2.0f * alpha)) * input[i])) + (alpha * input[(i + 1)]));
}



## 3. Generate target backends from the CrossGL IR

Now we take the generated `.cgl` file and ask CrossTL to produce other targets.

For a post, **CUDA → Metal** is the strongest visual demo because it makes the portability story obvious.

In [10]:
targets = {
    "metal": "outputs/steppy.metal",
    "directx": "outputs/steppy.hlsl",
    "opengl": "outputs/steppy.glsl",
    "mojo": "outputs/steppy.mojo",
    "rust": "outputs/steppy.rs",
}

generated = {}

for backend, output_path in targets.items():
    try:
        code = crosstl.translate("outputs/steppy.cgl", backend=backend, save_shader=output_path)
        generated[backend] = output_path
        print(f"✅ {backend:8s} → {output_path}")
    except Exception as e:
        print(f"⚠️ {backend:8s} skipped: {e}")

print("\nGenerated files:")
for backend, output_path in generated.items():
    print(f"- {backend}: {output_path}")

✅ metal    → outputs/steppy.metal
✅ directx  → outputs/steppy.hlsl
✅ opengl   → outputs/steppy.glsl


✅ mojo     → outputs/steppy.mojo
✅ rust     → outputs/steppy.rs

Generated files:
- metal: outputs/steppy.metal
- directx: outputs/steppy.hlsl
- opengl: outputs/steppy.glsl
- mojo: outputs/steppy.mojo
- rust: outputs/steppy.rs


## 4. Preview the Metal output

This is a visual output cell.

In [11]:
from pathlib import Path

metal_path = Path("outputs/steppy.metal")

if metal_path.exists():
    metal_code = metal_path.read_text()
    print(metal_code[:3000])
else:
    print("No Metal output found yet. Check the previous cell output.")


#include <metal_stdlib>
using namespace metal;

// Compute Shader
kernel void kernel_heat_step_1d(device float* input [[buffer(0)]], device float* output [[buffer(1)]], int n, float alpha, uint3 thread_position_in_grid [[thread_position_in_grid]], uint3 thread_position_in_threadgroup [[thread_position_in_threadgroup]], uint3 threadgroup_position_in_grid [[threadgroup_position_in_grid]], uint3 threads_per_threadgroup [[threads_per_threadgroup]]) {
    __attribute__((unused)) uint3 thread_id = thread_position_in_grid;
    __attribute__((unused)) uint3 block_id = threadgroup_position_in_grid;
    __attribute__((unused)) uint3 thread_local_id = thread_position_in_threadgroup;
    __attribute__((unused)) uint3 block_dim = threads_per_threadgroup;
    int i = threadgroup_position_in_grid.x * threads_per_threadgroup.x + thread_position_in_threadgroup.x;
    if (i >= n) {
        return;
    }
    if (i == 0 || i == n - 1) {
        output[i] = input[i];
        return;
    }
    output[i] = 

---

### Notes

This notebook is intentionally small for a social demo. For production use, always validate the generated backend output with the target platform toolchain and review any manual migration notes, especially for host/runtime integration.